In [0]:
import mlflow
import pandas as pd
from mlflow.models import infer_signature

class RoamRadarAgent(mlflow.pyfunc.PythonModel):
    def predict(self, context, model_input):
        lat = model_input["lat"].iloc[0]
        lon = model_input["lon"].iloc[0]
        meeting_soon = model_input["meeting_soon"].iloc[0]
        
        current_signal = 45 
        
        # We now return a Pandas DataFrame instead of a plain dictionary 
        # to ensure compatibility with Unity Catalog's strict schema enforcement.
        if meeting_soon and current_signal < 50:
            return pd.DataFrame([{
                "action": "notify",
                "message": "⚠️ Poor signal detected. Your Teams call starts in 5 mins—walk 50ft to the Drillfield for stable Wi-Fi.",
                "suggested_lat": 37.2276,
                "suggested_lon": -80.4221
            }])
            
        return pd.DataFrame([{
            "action": "ok", 
            "message": "Signal is stable.",
            "suggested_lat": 0.0,
            "suggested_lon": 0.0
        }])

# 1. Create a dummy dataframe mimicking the live GPS data your frontend will send
input_example = pd.DataFrame({
    "lat": [37.2284],
    "lon": [-80.4234],
    "meeting_soon": [True]
})

# 2. Create a dummy dataframe mimicking the agent's response
output_example = pd.DataFrame([{
    "action": "notify",
    "message": "Test message",
    "suggested_lat": 37.2276,
    "suggested_lon": -80.4221
}])

# 3. Generate the required metadata signature for Unity Catalog
signature = infer_signature(model_input=input_example, model_output=output_example)

# 4. Log the model, attaching both the signature and the input example
with mlflow.start_run():
    mlflow.pyfunc.log_model(
        artifact_path="agent_model", 
        python_model=RoamRadarAgent(),
        registered_model_name="workspace.default.roamradaragent",
        signature=signature,
        input_example=input_example
    )